# 🧪 Bộ Test Chuẩn — LocateAnything-3B trên 3 bài toán đếm

**Bật GPU T4 + Internet → Run All.**

Notebook này kiểm thử mô hình `nvidia/LocateAnything-3B` (open-vocabulary
detector) như một **hệ thống đếm** trên 3 bài toán thực tế, dùng chung một
đường ống `detect → track (ByteTrack) → đếm cắt vạch (LineZone)`:

| # | Bài toán | prompt | Vạch đếm |
|---|----------|--------|----------|
| 1 | 🚶 Đếm người **ra / vào** toà nhà | `person` | ngang, tách 2 chiều Vào/Ra |
| 2 | 📦 Đếm **sản phẩm** trên băng chuyền | `object` | dọc, chặn dòng chảy ngang |
| 3 | 🚗 Đếm **phương tiện** qua lại | `car` | ngang, nửa dưới khung |

Kết quả cuối là một **scorecard** so sánh cả 3 bài toán (số đếm mỗi chiều,
mật độ phát hiện, tốc độ FPS).

> 📁 Bộ code này còn có bản thư viện `la_counting/` + **43 unit test pytest**
> (chạy KHÔNG cần GPU) trong cùng repo — xem `README_TESTS.md`.

### Cách dùng
1. Chạy hết các cell cài đặt & định nghĩa.
2. Ở **Cell cấu hình video**, trỏ đường dẫn cho *people* / *conveyor* tới video
   bạn upload (video *vehicles* tự tải). Bài nào thiếu video sẽ được bỏ qua.
3. Xem scorecard ở cuối.


In [ ]:
# Cell 1 — Cài thư viện (transformers GHIM 4.57.1 theo README của NVIDIA)
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install -q -U "numpy<2.0.0" "transformers==4.57.1" "opencv-python-headless==4.11.0.86" \
    "Pillow==11.1.0" "decord==0.6.0" "lmdb==1.7.5" accelerate peft "supervision>=0.21" matplotlib
print("✅ Đã cài xong. Nếu Kaggle yêu cầu restart kernel, cứ Run All lại từ đầu.")

In [ ]:
# Cell 2 — Imports + kiểm tra GPU
import os, re, time, glob, shutil, urllib.request
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Sequence

import numpy as np
import cv2
import torch
import supervision as sv
import transformers
from PIL import Image

print(f"transformers: {transformers.__version__}   (kỳ vọng 4.57.1)")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU         : {p.name} | {p.total_memory/1024**3:.1f}GB | sm_{p.major}{p.minor}")

In [ ]:
# Cell 3 — Detection + hàm parse text của model ra bbox (thuần, đã unit-test)
NORM_SCALE = 1000
_RE_BOX = re.compile(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>")
_RE_BRK = re.compile(r"\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]")
_RE_LOC = re.compile(r"<loc_(\d+)>")

@dataclass
class Detection:
    bbox: Tuple[int, int, int, int]
    class_name: str
    confidence: float

def _add(dets, x1, y1, x2, y2, w, h, cls, conf, scale):
    x1, x2 = int(x1/scale*w), int(x2/scale*w)
    y1, y2 = int(y1/scale*h), int(y2/scale*h)
    if x2 > x1 and y2 > y1:
        dets.append(Detection((x1, y1, x2, y2), cls, conf))

def parse_boxes(text, class_name, w, h, default_conf=0.85):
    """3 định dạng, thử lần lượt; định dạng sau chỉ dùng nếu định dạng trước rỗng."""
    dets = []
    for m in _RE_BOX.findall(text):                       # <box><..></box> (0..1000)
        x1, y1, x2, y2 = (int(v) for v in m)
        _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    if not dets:                                          # [x1,y1,x2,y2] (tự đoán thang)
        for m in _RE_BRK.findall(text):
            v = [float(x) for x in m]
            sc = NORM_SCALE if max(v) > 1.5 else 1
            _add(dets, v[0], v[1], v[2], v[3], w, h, class_name, default_conf, sc)
    if not dets:                                          # <loc_n> theo bộ 4
        locs = _RE_LOC.findall(text)
        for i in range(0, len(locs)-3, 4):
            x1, y1, x2, y2 = (int(v) for v in locs[i:i+4])
            _add(dets, x1, y1, x2, y2, w, h, class_name, default_conf, NORM_SCALE)
    return dets

In [ ]:
# Cell 4 — Lớp bọc mô hình LocateAnythingDetector
class LocateAnythingDetector:
    def __init__(self, model_dir, max_new_tokens=1024):
        self.model_dir = model_dir
        self.max_new_tokens = max_new_tokens
        self._loaded = False
        self.dtype = torch.float16   # T4 (Turing) không có bfloat16 kernel

    def load(self):
        from transformers import AutoTokenizer, AutoProcessor, AutoConfig, AutoModel
        t0 = time.time()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir, trust_remote_code=True)
        self.processor = AutoProcessor.from_pretrained(self.model_dir, trust_remote_code=True)
        config = AutoConfig.from_pretrained(self.model_dir, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(
            self.model_dir, config=config, trust_remote_code=True,
            torch_dtype=self.dtype, device_map="auto", attn_implementation="sdpa")
        self.model.eval()
        self._loaded = True
        print(f"✅ Loaded in {time.time()-t0:.1f}s")
        if torch.cuda.is_available():
            print(f"   GPU Mem: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
        return self

    def _prep_input(self, v):
        if isinstance(v, np.ndarray):
            v = torch.from_numpy(v)
        if torch.is_tensor(v):
            if v.is_floating_point():
                return v.to(device=self.model.device, dtype=torch.float16)
            return v.to(self.model.device)
        return v

    def detect_pil(self, pil_image, prompt, max_new_tokens=None):
        if not self._loaded: self.load()
        w, h = pil_image.size
        max_tok = max_new_tokens or self.max_new_tokens
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": f"Locate all instances of: {prompt}"}]}]
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=text_prompt, images=[pil_image], return_tensors="pt")
        inputs = {k: self._prep_input(v) for k, v in inputs.items()}
        with torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=max_tok,
                                         do_sample=False, use_cache=True, tokenizer=self.tokenizer)
        if isinstance(output, (list, tuple)) and hasattr(output[0], "shape"):
            raw = self.tokenizer.decode(output[0], skip_special_tokens=True)
        elif hasattr(output, "shape"):
            raw = self.tokenizer.decode(output, skip_special_tokens=True)
        else:
            raw = str(output)
        return parse_boxes(raw, prompt, w, h), raw

    def detect_frame(self, bgr_frame, prompt, max_new_tokens=None):
        pil = Image.fromarray(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB))
        return self.detect_pil(pil, prompt, max_new_tokens)

In [ ]:
# Cell 5 — Định nghĩa 3 bài toán + pipeline đếm + scorecard
@dataclass(frozen=True)
class LineConfig:
    orientation: str      # "horizontal" | "vertical"
    position: float       # tỉ lệ 0..1
    anchor: str = "CENTER"
    def points(self, w, h):
        if self.orientation == "horizontal":
            y = int(self.position*h); return (0, y), (w, y)
        x = int(self.position*w); return (x, 0), (x, h)

@dataclass(frozen=True)
class Scenario:
    key: str; title: str; prompt: str; line: LineConfig
    resolution: Tuple[int, int] = (1280, 720)
    in_label: str = "IN"; out_label: str = "OUT"; max_frames: int = 150
    expect_min_crossings: int = 1

PEOPLE_IN_OUT = Scenario("people", "Đếm người ra/vào toà nhà", "person",
    LineConfig("horizontal", 0.50, "BOTTOM_CENTER"), in_label="Vào", out_label="Ra")
CONVEYOR = Scenario("conveyor", "Đếm sản phẩm trên băng chuyền", "object",
    LineConfig("vertical", 0.50, "CENTER"), in_label="Qua vạch", out_label="Ngược")
VEHICLES = Scenario("vehicles", "Đếm phương tiện qua lại", "car",
    LineConfig("horizontal", 0.55, "BOTTOM_CENTER"), in_label="Chiều tới", out_label="Chiều lui")
SCENARIOS = {s.key: s for s in (PEOPLE_IN_OUT, CONVEYOR, VEHICLES)}

def build_line_zone(scn, w=None, h=None):
    if w is None or h is None: w, h = scn.resolution
    (sx, sy), (ex, ey) = scn.line.points(w, h)
    return sv.LineZone(start=sv.Point(sx, sy), end=sv.Point(ex, ey),
                       triggering_anchors=(getattr(sv.Position, scn.line.anchor),))

def detections_to_sv(dets):
    if not dets: return sv.Detections.empty()
    return sv.Detections(
        xyxy=np.array([d.bbox for d in dets], dtype=np.float32),
        confidence=np.array([d.confidence for d in dets], dtype=np.float32),
        class_id=np.zeros(len(dets), dtype=int))

@dataclass
class CountResult:
    scenario_key: str; frames: int = 0; in_count: int = 0; out_count: int = 0
    total_detections: int = 0; elapsed_s: float = 0.0
    in_label: str = "IN"; out_label: str = "OUT"
    @property
    def total_crossings(self): return self.in_count + self.out_count
    @property
    def avg_detections(self): return self.total_detections/self.frames if self.frames else 0.0
    @property
    def fps(self): return self.frames/self.elapsed_s if self.elapsed_s > 0 else 0.0

class CountingPipeline:
    def __init__(self, detector, scenario, frame_rate=30):
        self.detector = detector; self.scenario = scenario
        self.w, self.h = scenario.resolution
        self.tracker = sv.ByteTrack(frame_rate=frame_rate)
        self.line_zone = build_line_zone(scenario, self.w, self.h)
        self.box_ann = sv.BoxAnnotator(thickness=2)
        self.label_ann = sv.LabelAnnotator(text_scale=0.5)
        self.trace_ann = sv.TraceAnnotator(thickness=2, trace_length=20)
        self.line_ann = sv.LineZoneAnnotator(thickness=3, text_scale=1.0)
        self.result = CountResult(scenario.key, in_label=scenario.in_label, out_label=scenario.out_label)

    def process_frame(self, frame_bgr):
        if frame_bgr.shape[1::-1] != (self.w, self.h):
            frame_bgr = cv2.resize(frame_bgr, (self.w, self.h))
        dets, raw = self.detector.detect_frame(frame_bgr, self.scenario.prompt)
        self.result.total_detections += len(dets)
        sv_d = self.tracker.update_with_detections(detections_to_sv(dets))
        self.line_zone.trigger(sv_d)
        self.result.frames += 1
        self.result.in_count = int(self.line_zone.in_count)
        self.result.out_count = int(self.line_zone.out_count)
        return frame_bgr, sv_d

    def annotate(self, frame_bgr, sv_d):
        a = self.trace_ann.annotate(frame_bgr.copy(), sv_d)
        a = self.box_ann.annotate(a, sv_d)
        if len(sv_d) and sv_d.tracker_id is not None:
            a = self.label_ann.annotate(a, sv_d, [f"#{t}" for t in sv_d.tracker_id])
        a = self.line_ann.annotate(a, self.line_zone)
        cv2.putText(a, f"{self.result.in_label}:{self.result.in_count}  {self.result.out_label}:{self.result.out_count}",
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        return a

def run_scenario(detector, scn, video_path, out_dir="/kaggle/working", max_frames=None, save_video=True):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"   ❌ Không mở được video: {video_path}"); return None
    limit = max_frames or scn.max_frames
    pipe = CountingPipeline(detector, scn, frame_rate=int(cap.get(cv2.CAP_PROP_FPS) or 30))
    out_path = os.path.join(out_dir, f"{scn.key}_counting.mp4")
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 30, (pipe.w, pipe.h)) if save_video else None
    t0 = time.time(); i = 0
    while i < limit:
        ret, frame = cap.read()
        if not ret: break
        frame, sv_d = pipe.process_frame(frame)
        if writer is not None: writer.write(pipe.annotate(frame, sv_d))
        i += 1
    pipe.result.elapsed_s = time.time() - t0
    cap.release()
    if writer is not None: writer.release(); print(f"   💾 {out_path}")
    return pipe.result

def print_scorecard(results):
    print("\n" + "="*76)
    print("📊 SCORECARD — LocateAnything-3B counting benchmark")
    print("="*76)
    print(f"{'scenario':10} {'frames':>6} {'IN':>5} {'OUT':>5} {'total':>6} {'det/frame':>10} {'fps':>7}  status")
    print("-"*76)
    for k, r in results.items():
        if r is None:
            print(f"{k:10} {'—':>6}  (bỏ qua: thiếu video)"); continue
        ok = r.total_crossings >= SCENARIOS[k].expect_min_crossings
        print(f"{k:10} {r.frames:>6} {r.in_count:>5} {r.out_count:>5} {r.total_crossings:>6} "
              f"{r.avg_detections:>10.2f} {r.fps:>7.2f}  {'✅ OK' if ok else '⚠️ KIỂM TRA'}")
    print("="*76)
    print("IN/OUT = 2 chiều cắt vạch — đổi nghĩa tuỳ bố trí camera.")
print("✅ Đã định nghĩa scenarios + pipeline + scorecard.")

In [ ]:
# Cell 6 — SMOKE TEST (KHÔNG cần model): kiểm tra đường ống đếm chạy đúng
# Dùng detector giả di chuyển vật thể theo kịch bản -> phải ra đúng số lần cắt vạch.
class _ScriptedDetector:
    def __init__(self, frames_boxes): self.frames_boxes = frames_boxes; self.i = 0
    def detect_frame(self, frame, prompt, max_new_tokens=None):
        boxes = self.frames_boxes[self.i] if self.i < len(self.frames_boxes) else []
        self.i += 1
        return [Detection(tuple(int(round(v)) for v in b), prompt, 0.85) for b in boxes], "fake"

def _linear(a, b, n): return [tuple(s+(e-s)*i/max(1,n-1) for s, e in zip(a, b)) for i in range(n)]
_N = 26; _blank = [np.zeros((720,1280,3), np.uint8) for _ in range(_N)]

def _run_fake(scn, script):
    pipe = CountingPipeline(_ScriptedDetector(script), scn)
    for f in _blank[:len(script)]:
        fr, d = pipe.process_frame(f)
    return pipe.result

# người đi xuống (cắt vạch ngang), sản phẩm sang phải (cắt vạch dọc), 2 xe đi xuống
_r1 = _run_fake(PEOPLE_IN_OUT, [[b] for b in _linear((600,150,680,190),(600,470,680,510), _N)])
_r2 = _run_fake(CONVEYOR,      [[b] for b in _linear((300,300,380,380),(1000,300,1080,380), _N)])
_t1 = _linear((300,150,400,220),(300,470,400,540), _N); _t2 = _linear((800,150,900,220),(800,470,900,540), _N)
_r3 = _run_fake(VEHICLES, [[_t1[i], _t2[i]] for i in range(_N)])

_checks = [("people 1 cross", _r1.total_crossings == 1), ("conveyor 1 cross", _r2.total_crossings == 1),
           ("vehicles 2 cross", _r3.total_crossings == 2)]
for name, ok in _checks: print(f"  {'✅' if ok else '❌'} {name}")
assert all(ok for _, ok in _checks), "Smoke test FAIL — pipeline đếm sai!"
print("🎉 SMOKE TEST PASSED — đường ống detect→track→count hoạt động đúng.")

In [ ]:
# Cell 7 — Tải model + patch bfloat16->float16 cho T4
from huggingface_hub import snapshot_download
MODEL_ID = "nvidia/LocateAnything-3B"
print("📥 Downloading model... (lần đầu ~6GB)")
model_dir = snapshot_download(MODEL_ID)
mc = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
if os.path.exists(mc): shutil.rmtree(mc)
f = os.path.join(model_dir, "modeling_locateanything.py")
if os.path.exists(f):
    real_f = os.path.realpath(f); code_txt = open(real_f).read()
    old = "pixel_values = pixel_values.to(self.language_model.dtype)"
    if old in code_txt:
        open(real_f, "w").write(code_txt.replace(old, "pixel_values = pixel_values.to(torch.float16)  # T4 fix"))
        print("✅ Patched bfloat16 -> float16")
print(f"📁 {model_dir}\n✅ Ready!")

In [ ]:
# Cell 8 — Nạp mô hình
detector = LocateAnythingDetector(model_dir=model_dir, max_new_tokens=1024)
detector.load()

In [ ]:
# Cell 9 — CẤU HÌNH VIDEO CHO TỪNG BÀI TOÁN
# 👉 Sửa đường dẫn people/conveyor tới video BẠN upload. Video vehicles tự tải.
VIDEO_PATHS = {
    "people":   "/kaggle/input/your-dataset/people.mp4",     # <-- SỬA
    "conveyor": "/kaggle/input/datasets/huyhung2004/couting/istockphoto-2179809582-640_adpp_is.mp4",  # <-- SỬA nếu cần
    "vehicles": "/kaggle/working/vehicles.mp4",               # tự tải bên dưới
}
MAX_FRAMES = 150   # tăng nếu muốn video dài hơn

# Tự tải video giao thông mẫu (Roboflow) nếu chưa có
if not os.path.exists(VIDEO_PATHS["vehicles"]):
    print("📥 Tải video giao thông mẫu...")
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/vehicles.mp4",
                               VIDEO_PATHS["vehicles"])
for k, p in VIDEO_PATHS.items():
    print(f"  {'✅' if os.path.exists(p) else '❌ THIẾU'}  {k:9} -> {p}")

In [ ]:
# Cell 10 — CHẠY CẢ 3 BÀI TOÁN + IN SCORECARD
results = {}
for key, scn in SCENARIOS.items():
    print(f"\n▶️  {scn.title}  (prompt={scn.prompt!r})")
    path = VIDEO_PATHS.get(key)
    if not path or not os.path.exists(path):
        print(f"   ⏭️  Bỏ qua: không tìm thấy video."); results[key] = None; continue
    results[key] = run_scenario(detector, scn, path, max_frames=MAX_FRAMES)

print_scorecard(results)

In [ ]:
# Cell 11 — Xem video kết quả (chuyển sang H.264 để hiển thị)
from IPython.display import Video, display
for key, r in results.items():
    if r is None: continue
    src = f"/kaggle/working/{key}_counting.mp4"
    dst = src.replace(".mp4", "_h264.mp4")
    os.system(f"ffmpeg -y -i {src} -vcodec libx264 {dst} 2>/dev/null")
    if os.path.exists(dst):
        print(f"🎬 {SCENARIOS[key].title}")
        display(Video(dst, embed=True, width=820))

## (Tùy chọn) Đánh giá độ chính xác phát hiện trên dataset ảnh có nhãn

Ba bài trên đo **năng lực đếm** (đếm cắt vạch). Nếu bạn có dataset ảnh gán nhãn
theo định dạng YOLO (`images/*.jpg` + `labels/*.txt`), cell dưới đo thêm
**Precision / Recall / mAP@50** cho khả năng khoanh vùng của model.


In [ ]:
# Cell 12 — (tùy chọn) mAP / Precision / Recall trên dataset ảnh YOLO
from tqdm.auto import tqdm
BASE = "/kaggle/input/datasets/huyhung2004/package/test"   # <-- SỬA đường dẫn dataset
EVAL_PROMPT = "cardboard box"

image_paths = sorted(glob.glob(f"{BASE}/images/*.jpg")) or sorted(glob.glob(f"{BASE}/*.jpg"))
if not image_paths:
    print("⏭️  Không tìm thấy dataset ảnh — bỏ qua phần đánh giá mAP.")
else:
    targets, predictions = [], []
    for img_p in tqdm(image_paths, desc="eval"):
        frame = cv2.imread(img_p)
        if frame is None: continue
        h, w = frame.shape[:2]
        lbl_p = img_p.replace("/images/", "/labels/").replace(".jpg", ".txt")
        gt = []
        if os.path.exists(lbl_p):
            for line in open(lbl_p):
                parts = line.split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = map(float, parts[1:5])
                    gt.append([(xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h])
        targets.append(sv.Detections(xyxy=np.array(gt, np.float32), class_id=np.zeros(len(gt), int))
                       if gt else sv.Detections.empty())
        dets, _ = detector.detect_frame(cv2.resize(frame, (1280, 720)), EVAL_PROMPT)
        if dets:
            boxes = [[d.bbox[0]*w/1280, d.bbox[1]*h/720, d.bbox[2]*w/1280, d.bbox[3]*h/720] for d in dets]
            predictions.append(sv.Detections(xyxy=np.array(boxes, np.float32),
                confidence=np.array([d.confidence for d in dets], np.float32),
                class_id=np.zeros(len(dets), int)))
        else:
            predictions.append(sv.Detections.empty())

    mAP = sv.MeanAveragePrecision.from_detections(predictions=predictions, targets=targets)
    print(f"\nmAP@50    : {mAP.map50:.3f}")
    print(f"mAP@50-95 : {mAP.map50_95:.3f}")
    print("Lưu ý: model chỉ phát 1 confidence cố định (0.85) nên đường PR bị suy biến — "
          "coi mAP@50 như tham khảo, số đếm cắt vạch mới là chỉ tiêu chính.")

---
### Ghi chú
- Nếu chiều **Vào/Ra** (hoặc Tới/Lui) bị ngược so với thực tế, chỉ cần đổi
  `in_label`/`out_label` của scenario tương ứng ở **Cell 5** — logic không đổi.
- Muốn đổi đối tượng đếm: sửa `prompt` (vd `person`→`worker`, `car`→`truck`,
  `object`→`bottle`/`box`).
- Đặt lại vị trí vạch qua `LineConfig(position=...)` (tỉ lệ 0..1).
- Bản thư viện đã test (`la_counting/` + `tests/`, 43 unit test) nằm cùng repo,
  chạy `pytest` KHÔNG cần GPU — xem `README_TESTS.md`.
